# 🧠 Machine Learning — v8
### Figuras  ·  Con db de TODOS los parámetro (nº=79)
**Targets:** `reporter` · `genome` · `medium`  
**Datos:** `dbm_v8.xlsx`

---
## 0. Imports y configuración global

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, roc_curve, auc
)
from sklearn.ensemble import (
    AdaBoostClassifier, RandomForestClassifier,
    GradientBoostingClassifier, ExtraTreesClassifier
)
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from itertools import cycle

try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
except ImportError:
    XGBOOST_OK = False
    print("⚠️  XGBoost no instalado")

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_OK = True
except ImportError:
    LIGHTGBM_OK = False
    print("⚠️  LightGBM no instalado")

---
## 1. Estilos globales — POSTER
> Letra grande, colores vivos, sin espacio muerto

In [ ]:
# ── Paleta y colores ──────────────────────────────────────────────────────────
PALETTE    = ["#264653", "#2a9d8f", "#e9c46a", "#f4a261", "#e76f51",
              "#457b9d", "#a8dadc", "#e63946", "#606c38", "#dda15e"]
CMAP_CM    = "Blues"
CMAP_FIMP  = "YlOrRd"
BG_COLOR   = "#FFFFFF"
ACCENT     = "#2a9d8f"

# ── Tamaños POSTER (todo grande) ──────────────────────────────────────────────
FONT_TITLE  = 18
FONT_LABEL  = 15
FONT_TICK   = 13
FONT_LEGEND = 13
FONT_ANNOT  = 12

plt.rcParams.update({
    "figure.facecolor":  BG_COLOR,
    "axes.facecolor":    BG_COLOR,
    "font.family":       "DejaVu Sans",
    "font.size":         FONT_TICK,
    "axes.titlesize":    FONT_TITLE,
    "axes.labelsize":    FONT_LABEL,
    "xtick.labelsize":   FONT_TICK,
    "ytick.labelsize":   FONT_TICK,
    "legend.fontsize":   FONT_LEGEND,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})

# ── Idioma y carpetas ─────────────────────────────────────────────────────────
LANG    = "ENG"   # ← cambia a "ESP" para español
carpeta = f"Figuras_v8_param{LANG}"
sufijo  = f"_{LANG}"
os.makedirs(carpeta, exist_ok=True)
print(f"✓ Carpeta lista: /{carpeta}/")

---
## 2. Carga de datos

In [ ]:
df = pd.read_excel("dbm_v8.xlsx", index_col=[0, 1]).reset_index()   #v8 ES CON LOS PARAMETROS FINALES DE COSENO, DERIVADAS Y WL
TARGETS = ["reporter", "genome", "medium"]   # ← targets disponibles
FEAT_START = 7                                # columnas de features desde aquí
print(f"Shape: {df.shape}")
print(f"Columnas target: {TARGETS}")
df.head(3)

---
## 3. Funciones base de entrenamiento

In [ ]:
def get_Xy(df, target):
    """Prepara X numérico e y encodado."""
    y_raw = df[target]
    X_num = df.iloc[:, FEAT_START:].select_dtypes(include="number")
    X     = SimpleImputer(strategy="mean").fit_transform(X_num)
    le    = LabelEncoder()
    y     = le.fit_transform(y_raw)
    return X, y, le, X_num


def entrenar_rf(df, target):
    """Entrena Random Forest y devuelve diccionario con todo."""
    X, y, le, X_num = get_Xy(df, target)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    modelo = RandomForestClassifier(
        n_estimators=100, random_state=42, class_weight="balanced"
    )
    modelo.fit(X_train, y_train)
    y_pred  = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)
    acc     = accuracy_score(y_test, y_pred)
    return dict(modelo=modelo, le=le, clases=list(le.classes_),
                X_num=X_num, X_train=X_train, X_test=X_test,
                y_train=y_train, y_test=y_test, y_pred=y_pred,
                y_proba=y_proba, acc=acc,
                feat_imp=pd.Series(modelo.feature_importances_, index=X_num.columns))


def entrenar_light(df, target):
    """Entrena LightGBM y devuelve diccionario con todo."""
    X, y, le, X_num = get_Xy(df, target)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    modelo = LGBMClassifier(
        n_estimators=100, random_state=42, class_weight="balanced", verbose=-1
    )
    modelo.fit(X_train, y_train)
    y_pred  = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)
    acc     = accuracy_score(y_test, y_pred)
    return dict(modelo=modelo, le=le, clases=list(le.classes_),
                X_num=X_num, X_train=X_train, X_test=X_test,
                y_train=y_train, y_test=y_test, y_pred=y_pred,
                y_proba=y_proba, acc=acc,
                feat_imp=pd.Series(modelo.feature_importances_, index=X_num.columns))


def entrenar_todos_modelos(df, target):
    """
    Entrena todos los clasificadores sobre un target.
    Devuelve dict {nombre: {acc, modelo, y_pred, y_proba, y_test, le}}
    """
    X_num = df.iloc[:, FEAT_START:].select_dtypes(include="number")
    X     = X_num.fillna(0)
    le    = LabelEncoder()
    y     = le.fit_transform(df[target])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    modelos = [
        ("AdaBoost",           AdaBoostClassifier(n_estimators=100, random_state=42)),
        ("RandomForest",       RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")),
        ("GradBoost",          GradientBoostingClassifier(n_estimators=100, random_state=42)),
        ("ExtraTrees",         ExtraTreesClassifier(n_estimators=100, random_state=42)),
        ("SVM",                SVC(random_state=42, probability=True)),
        ("KNN",                KNeighborsClassifier(n_neighbors=5)),
        ("LogReg",             LogisticRegression(max_iter=1000, random_state=42)),
        ("MLP",                MLPClassifier(max_iter=500, random_state=42)),
    ]
    if XGBOOST_OK:
        modelos.append(("XGBoost",  XGBClassifier(n_estimators=100, random_state=42, verbosity=0)))
    if LIGHTGBM_OK:
        modelos.append(("LightGBM", LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)))

    resultados = {}
    for nombre, m in modelos:
        m.fit(X_train, y_train)
        y_pred  = m.predict(X_test)
        y_proba = m.predict_proba(X_test)
        acc     = accuracy_score(y_test, y_pred)
        resultados[nombre] = dict(acc=acc, modelo=m, y_pred=y_pred,
                                  y_proba=y_proba, y_test=y_test, le=le)
    return resultados



# ─── NUEVO: registro de modelos y función genérica ────────────────────────────
def _build_modelo(nombre):
    """Factory: crea una instancia nueva del modelo por nombre."""
    if nombre == "AdaBoost":     return AdaBoostClassifier(n_estimators=100, random_state=42)
    if nombre == "RandomForest": return RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced")
    if nombre == "GradBoost":    return GradientBoostingClassifier(n_estimators=100, random_state=42)
    if nombre == "ExtraTrees":   return ExtraTreesClassifier(n_estimators=100, random_state=42)
    if nombre == "SVM":          return SVC(random_state=42, probability=True)
    if nombre == "KNN":          return KNeighborsClassifier(n_neighbors=5)
    if nombre == "LogReg":       return LogisticRegression(max_iter=1000, random_state=42)
    if nombre == "MLP":          return MLPClassifier(max_iter=500, random_state=42)
    if nombre == "XGBoost":      
        if not XGBOOST_OK: raise ImportError("xgboost no instalado")
        return XGBClassifier(n_estimators=100, random_state=42, verbosity=0)
    if nombre == "LightGBM":     
        if not LIGHTGBM_OK: raise ImportError("lightgbm no instalado")
        return LGBMClassifier(n_estimators=100, random_state=42, class_weight="balanced", verbose=-1)
    raise ValueError(f"Modelo desconocido: {nombre}. Opciones: AdaBoost, RandomForest, GradBoost, "
                     f"ExtraTrees, SVM, KNN, LogReg, MLP, XGBoost, LightGBM")


def entrenar_modelo(df, target, nombre_modelo):
    """
    Entrena CUALQUIERA de los 10 clasificadores y devuelve el mismo diccionario
    que entrenar_rf/entrenar_light, con feat_imp calculada según el modelo:
      - Modelos con .feature_importances_ nativo (todos los árboles/boosting):
        se usa directamente.
      - Modelos lineales (LogReg, SVM lineal): se usan los valores absolutos de
        los coeficientes promediados sobre clases.
      - Modelos sin importancia nativa (SVM-RBF, KNN, MLP): se calcula
        permutation_importance (tarda ~30-60s extra pero es la forma estándar).
    """
    from sklearn.inspection import permutation_importance
    
    X, y, le, X_num = get_Xy(df, target)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    modelo = _build_modelo(nombre_modelo)
    modelo.fit(X_train, y_train)
    y_pred  = modelo.predict(X_test)
    y_proba = modelo.predict_proba(X_test)
    acc     = accuracy_score(y_test, y_pred)
    
    # ── Feature importance según tipo de modelo ──────────────────────────────
    if hasattr(modelo, "feature_importances_"):
        feat_imp = pd.Series(modelo.feature_importances_, index=X_num.columns)
    elif hasattr(modelo, "coef_") and modelo.coef_.ndim >= 1:
        # Lineales (LogReg, SVM lineal): media de |coef_| sobre clases
        coef = modelo.coef_
        if coef.ndim == 1:
            feat_imp = pd.Series(np.abs(coef), index=X_num.columns)
        else:
            feat_imp = pd.Series(np.abs(coef).mean(axis=0), index=X_num.columns)
    else:
        # SVM-RBF / KNN / MLP: permutation importance
        print(f"  (calculando permutation_importance para {nombre_modelo}...)")
        pi = permutation_importance(modelo, X_test, y_test, n_repeats=10,
                                     random_state=42, n_jobs=-1)
        feat_imp = pd.Series(pi.importances_mean, index=X_num.columns)
    
    return dict(modelo=modelo, le=le, clases=list(le.classes_),
                X_num=X_num, X_train=X_train, X_test=X_test,
                y_train=y_train, y_test=y_test, y_pred=y_pred,
                y_proba=y_proba, acc=acc, feat_imp=feat_imp,
                nombre_modelo=nombre_modelo)

print("✓ Funciones listas (incluye entrenar_modelo genérico)")

---
## 4. Entrenar todos los modelos para todos los targets
> Esta celda tarda un poco — entrena todos los clasificadores × 3 targets

In [ ]:
# Diccionario: todos_res[target][nombre_modelo] = {acc, modelo, ...}
todos_res = {}
for t in TARGETS:
    print(f"\n── Entrenando target: {t.upper()} ──────────────")
    todos_res[t] = entrenar_todos_modelos(df, t)

# Tabla resumen de accuracy
nombres_modelos = list(todos_res[TARGETS[0]].keys())
df_acc = pd.DataFrame(
    {t: {n: todos_res[t][n]["acc"] for n in nombres_modelos} for t in TARGETS}
)
df_acc["mean"] = df_acc.mean(axis=1)
df_acc["std"]  = df_acc[TARGETS].std(axis=1)
print("\n── Accuracy por modelo y target ──")
print(df_acc.round(3).to_string())

---
## 5. PUNTO 2 — Barplot de Accuracy por modelo


### Opción B — Media ± std de los 3 targets

In [ ]:
# ── OPCIÓN B: barras horizontales, mejor arriba, peor abajo ───────────────────
df_sorted = df_acc.sort_values("mean", ascending=True)  # ascending=True → mejor queda arriba en barh
n         = len(df_sorted)

bar_cols   = plt.cm.get_cmap("YlGn")(np.linspace(0.35, 0.85, n))  # mejor = más oscuro arriba

sns.set_style("whitegrid")
plt.rc('axes',edgecolor='k')


fig, ax = plt.subplots(figsize=(6, 5), facecolor=BG_COLOR)
ax.set_yticks([])

bars = ax.barh(
    df_sorted.index, df_sorted["mean"],
    color=bar_cols, edgecolor="white", height=0.6,
    xerr=df_sorted["std"], capsize=5,
    error_kw=dict(elinewidth=2, ecolor="#444", capthick=2)
)

for bar, (_, row) in zip(bars, df_sorted.iterrows()):
    ax.text(
        row["mean"] + row["std"] + 0.008,
        bar.get_y() + bar.get_height() / 2,
        f"{row['mean']:.3f}",
        va="center", ha="left",
        fontsize=FONT_ANNOT-3, fontweight="bold", color="#264653"
    )

for bar, nombre in zip(bars, df_sorted.index):
    ax.text(
        0.04,                              # ← margen desde el borde izquierdo
        bar.get_y() + bar.get_height() / 2,
        nombre,
        va="center", ha="left",
        fontsize=FONT_TICK - 2,
        color="white",
        fontweight="bold"
    )

ax.set_xlabel("Accuracy (mean ± std)", fontsize=FONT_LABEL)
#ax.set_ylabel("", fontsize=FONT_LABEL)
#ax.set_ylabel()
ax.set_yticks([])
ax.set_xlim(0, 1.18)
ax.set_title("Model Accuracy — Mean ± Std across all targets",
             fontsize=FONT_TITLE-3, fontweight="bold", pad=14)

ax.axvline(df_sorted["mean"].mean(), color=ACCENT, ls="--", lw=1.8,
           label=f"Overall mean = {df_sorted['mean'].mean():.3f}")

ax.axvline(0.5, color="gray", ls=":", lw=1.2, label="Random (0.5)")
ax.tick_params(axis="y", labelsize=FONT_TICK)
ax.legend(fontsize=FONT_LEGEND-4, frameon=True, loc="lower right")

plt.tight_layout()

fname = f"{carpeta}/accuracy_barplot_mean_std{sufijo}.png"
plt.savefig(fname, dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {fname}")

### Opción C — Heatmap de accuracy (modelos × targets)

In [ ]:
# ── OPCIÓN C: Heatmap — modelos × targets, ordenado por accuracy media desc ──
df_heat = df_acc[TARGETS].copy()
df_heat["mean"] = df_acc["mean"]

# Ordenar TODOS los modelos por accuracy media descendente (mejor arriba)
df_heat = df_heat.sort_values("mean", ascending=False).drop(columns="mean")

fig, ax = plt.subplots(figsize=(8, 7), facecolor=BG_COLOR)

sns.heatmap(
    df_heat,
    annot=True, fmt=".3f",
    cmap="YlGn", vmin=0.4, vmax=1.0,
    linewidths=0.8, linecolor="white",
    annot_kws={"size": FONT_ANNOT, "weight": "bold"},
    cbar_kws={"shrink": 0.8, "label": "Accuracy"},
    ax=ax
)
ax.set_title("Accuracy Heatmap — models × targets\n(sorted by mean accuracy)",
             fontsize=FONT_TITLE, fontweight="bold", pad=14)
ax.set_xlabel("", fontsize=FONT_LABEL)
ax.set_ylabel("", fontsize=FONT_LABEL)
ax.set_xticklabels([t.capitalize() for t in TARGETS],
                   fontsize=FONT_TICK, rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=FONT_TICK, rotation=0)
plt.tight_layout()

fname = f"{carpeta}/accuracy_heatmap{sufijo}.png"
plt.savefig(fname, dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {fname}")

---
## 6. PUNTO 3 — ROC Curve: 1 curva por modelo, AUC promedio de los 3 targets

> **Cómo funciona:** para cada modelo, se calcula la curva ROC macro-avg sobre todos los targets juntos (binarizando todo en un único array), y se dibuja **una sola curva** por modelo. El eje X empieza en -0.02 para que se vea el origen.

### Opción A — Líneas simples, una por modelo

In [ ]:
def calcular_auc_promedio(todos_res, nombre_modelo, targets):
    """
    Calcula una única curva ROC para un modelo promediando los 3 targets.
    Estrategia: concatenar las predicciones de probabilidad de todos los targets
    y calcular una curva ROC binaria One-vs-Rest global.
    Devuelve fpr, tpr, auc_val.
    """
    all_fpr_list, all_tpr_list, all_auc_list = [], [], []

    for t in targets:
        res      = todos_res[t][nombre_modelo]
        y_test   = res["y_test"]
        y_proba  = res["y_proba"]
        n_clases = y_proba.shape[1]

        # OvR por clase → acumular
        y_bin = label_binarize(y_test, classes=list(range(n_clases)))
        for i in range(n_clases):
            fpr_i, tpr_i, _ = roc_curve(y_bin[:, i], y_proba[:, i])
            all_fpr_list.append(fpr_i)
            all_tpr_list.append(tpr_i)
            all_auc_list.append(auc(fpr_i, tpr_i))

    # Interpolar todas las curvas sobre una rejilla común
    mean_fpr = np.linspace(0, 1, 300)
    mean_tpr = np.zeros_like(mean_fpr)
    for fpr_i, tpr_i in zip(all_fpr_list, all_tpr_list):
        mean_tpr += np.interp(mean_fpr, fpr_i, tpr_i)
    mean_tpr /= len(all_fpr_list)
    auc_val   = float(np.mean(all_auc_list))

    return mean_fpr, mean_tpr, auc_val


# ── Calcular para todos los modelos ───────────────────────────────────────────
roc_data = {}
for nombre in nombres_modelos:
    fpr, tpr, auc_val = calcular_auc_promedio(todos_res, nombre, TARGETS)
    roc_data[nombre]  = (fpr, tpr, auc_val)

# Ordenar por AUC descendente
roc_sorted = sorted(roc_data.items(), key=lambda x: -x[1][2])

print("AUC promedio por modelo:")
for nombre, (_, _, a) in roc_sorted:
    print(f"  {nombre:<22} AUC = {a:.4f}")

In [ ]:
# ── OPCIÓN A: líneas simples ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8), facecolor=BG_COLOR)

colors = plt.cm.get_cmap("tab10")(np.linspace(0, 0.9, len(roc_sorted)))

for (nombre, (fpr, tpr, auc_val)), color in zip(roc_sorted, colors):
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f"{nombre:<18}  AUC = {auc_val:.3f}")

# Línea de clasificador aleatorio
ax.plot([-0.02, 1], [-0.02, 1], "k--", lw=1.8, label="Random classifier")

ax.set_xlim([-0.02, 1.0])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel("False Positive Rate (FPR)", fontsize=FONT_LABEL)
ax.set_ylabel("True Positive Rate (TPR)", fontsize=FONT_LABEL)
ax.set_title("ROC Curve — All models\n(AUC averaged across all targets)",
             fontsize=FONT_TITLE, fontweight="bold", pad=14)
ax.legend(fontsize=11, loc="lower right",
          frameon=True, framealpha=0.9,
          prop={"family": "monospace"})
ax.grid(alpha=0.2)
plt.tight_layout()

fname = f"{carpeta}/roc_all_models_lines{sufijo}.png"
plt.savefig(fname, dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {fname}")

### Opción B — Misma curva pero con área sombreada bajo cada línea

In [ ]:
# ── OPCIÓN B: curvas con área sombreada ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 8), facecolor=BG_COLOR)

#colors = plt.cm.get_cmap("tab10")(np.linspace(0, 0.9))
colors = sns.color_palette("Paired", 10)#plt.cm.get_cmap("tab10")(np.linspace(0, 0.9))

for (nombre, (fpr, tpr, auc_val)), color in zip(roc_sorted, colors):
    ax.plot(fpr, tpr, color=color, lw=2.5,
            label=f"{nombre:<18}  AUC = {auc_val:.3f}")
    #ax.scatter(np.mean(fpr), np.mean(fpr), color=color, lw=2.5)

ax.plot([-0.02, 1], [-0.02, 1], "k--", lw=1.8, label="Random classifier")

ax.set_xlim([-0.02, 1.0])
ax.set_ylim([-0.02, 1.02])
ax.set_xlabel("False Positive Rate (FPR)", fontsize=FONT_LABEL)
ax.set_ylabel("True Positive Rate (TPR)", fontsize=FONT_LABEL)
ax.set_title("ROC Curve — All models\n(AUC averaged across all targets)",
             fontsize=FONT_TITLE, fontweight="bold", pad=14)
ax.legend(fontsize=11, loc="lower right",
          frameon=True, framealpha=0.9,
          prop={"family": "monospace"})
ax.grid(alpha=0.2)
plt.tight_layout()

fname = f"{carpeta}/roc_all_models_filled{sufijo}.png"
plt.savefig(fname, dpi=300, bbox_inches="tight")
plt.show()
print(f"✓ Saved: {fname}")

---
## 7. PUNTO 4 — Confusion Matrix, Top 15 y Top 10 Feature Importances

> **Elige modelo y target aquí. Por defecto: RF.**  
> Cada figura se guarda por separado.

In [ ]:
# ── CONFIGURACIÓN ─────────────────────────────────────────────────────────────
# ⭐ Elige cualquier modelo. Opciones disponibles:
#    "AdaBoost", "RandomForest", "GradBoost", "ExtraTrees",
#    "SVM", "KNN", "LogReg", "MLP", "XGBoost", "LightGBM"
MODEL = "ExtraTrees"
# ─────────────────────────────────────────────────────────────────────────────

# Prefijo para nombrar archivos (lowercase del nombre del modelo)
_prefijo = MODEL.lower()

# Entrena los 3 targets con el modelo elegido
modelos_fit = {t: entrenar_modelo(df, t, MODEL) for t in TARGETS}
print(f"✓ Modelos entrenados con: {MODEL}")
for t in TARGETS:
    print(f"  {t:<12} Accuracy = {modelos_fit[t]['acc']:.2%}")

### 7.1 Confusion Matrix — normalizada por clase real (una figura por target)

In [ ]:
def plot_confusion_matrix(m, target, prefijo):
    clases        = m["clases"]
    cm            = confusion_matrix(m["y_test"], m["y_pred"])
    cm_norm       = confusion_matrix(m["y_test"], m["y_pred"], normalize="true")
    nombre_modelo = m.get("nombre_modelo", prefijo.capitalize())

    n  = len(clases)
    sz = max(6, n * 1.5)

    fig, ax = plt.subplots(figsize=(sz, sz * 0.85), facecolor=BG_COLOR)

    sns.heatmap(
        cm_norm, annot=True, fmt=".0%",
        cmap=CMAP_CM, vmin=0, vmax=1,
        xticklabels=clases, yticklabels=clases,
        linewidths=0.8, linecolor="white", square=True,
        annot_kws={"size": FONT_ANNOT + 2, "weight": "bold"},
        cbar_kws={"shrink": 0.75, "label": "Recall"},
        ax=ax
    )
    for i in range(n):
        for j in range(n):
            color_txt = "white" if cm_norm[i, j] > 0.55 else "#444"
            ax.text(j + 0.5, i + 0.72, f"n={cm[i,j]}",
                    ha="center", va="center",
                    fontsize=FONT_ANNOT - 1, color=color_txt)

    ax.set_title(
        f"Confusion Matrix · {nombre_modelo} · «{target}»\n"
        f"(normalized by true class — Acc {m['acc']:.1%})",
        fontsize=FONT_TITLE, fontweight="bold", pad=14
    )
    ax.set_xlabel("Predicted", fontsize=FONT_LABEL)
    ax.set_ylabel("True",      fontsize=FONT_LABEL)
    ax.set_xticklabels(
        clases, rotation=45, ha="right",
        rotation_mode="anchor", fontsize=FONT_TICK
    )
    ax.set_yticklabels(clases, rotation=0, fontsize=FONT_TICK)

    plt.tight_layout()
    fname = f"{carpeta}/{prefijo}_confusion_matrix_{target}{sufijo}.png"
    plt.savefig(fname, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {fname}")


for t in TARGETS:
    plot_confusion_matrix(modelos_fit[t], t, _prefijo)

In [ ]:
sns.color_palette("flare", 15)

### 7.2 Top 15 Feature Importances (una figura por target)

In [ ]:
import seaborn as sns  # añade este import arriba del notebook/script si no lo tienes ya

def plot_feature_importance(m, target, prefijo, top_n=15):
    feat_imp      = m["feat_imp"]
    nombre_modelo = m.get("nombre_modelo", prefijo.capitalize())
    top           = feat_imp.nlargest(top_n)

    # ── Paleta degradada "flare" ──────────────────────────────────────────────
    colors = sns.color_palette("flare", top_n)[::-1]

    fig, ax = plt.subplots(figsize=(10, 7), facecolor=BG_COLOR)
    bars = ax.barh(range(top_n), top.values, color=colors, edgecolor="white", height=0.7)
    for bar, val in zip(bars, top.values):
        ax.text(
            val + top.values.max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center",
            fontsize=FONT_ANNOT, color="#264653"
        )
    ax.set_yticks(range(top_n))
    ax.set_yticklabels(top.index, fontsize=FONT_TICK)
    ax.invert_yaxis()
    ax.set_xlabel("Importance", fontsize=FONT_LABEL)
    ax.set_title(
        f"Top {top_n} Feature Importances · {nombre_modelo} · «{target}»",
        fontsize=FONT_TITLE, fontweight="bold", pad=14
    )
    ax.axvline(top.values.mean(), color=ACCENT, ls="--",
               lw=2, label=f"Mean = {top.values.mean():.4f}")
    ax.legend(fontsize=FONT_LEGEND, frameon=True,
              loc="lower right", framealpha=0.9)
    plt.tight_layout()
    fname = f"{carpeta}/{prefijo}_top{top_n}_features_{target}{sufijo}.png"
    plt.savefig(fname, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {fname}")

# ── Top 15 ────────────────────────────────────────────────────────────────────
for t in TARGETS:
    plot_feature_importance(modelos_fit[t], t, _prefijo, top_n=15)

### 7.3 Top 10 Feature Importances (una figura por target)

In [ ]:
for t in TARGETS:
    plot_feature_importance(modelos_fit[t], t, _prefijo, top_n=10)

In [ ]:
### next steps